In [4]:
!pip install streamlit pyngrok --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 86.8 MB/s eta 0:00:00


In [14]:
%%writefile app_mall.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

st.set_page_config(page_title="Mall Customer Segmentation", layout="wide")

@st.cache_data
def load_data():
    df = pd.read_csv("Mall_Customers.csv")
    return df

def main():
    st.title(" Customer Segmentation menggunakan K-Means")
    st.write("Aplikasi ini mengelompokkan pelanggan berdasarkan Pendapatan Tahunan dan Skor Pengeluaran.")

    df = load_data()
    X = df.iloc[:, [3, 4]]


    st.sidebar.header("Konfigurasi Model")
    k_value = st.sidebar.slider("Pilih jumlah Cluster (k)", 2, 10, 5)

    # Tab Menu
    tab1, tab2, tab3 = st.tabs(["📊 Dataset", "📉 Elbow Method", "🎯 Hasil Clustering"])

    with tab1:
        st.subheader("Data Mentah")
        st.dataframe(df, use_container_width=True)
        st.subheader("Statistik Deskriptif")
        st.write(df.describe())

    with tab2:
        st.subheader("The Elbow Method")
        st.write("Metode ini digunakan untuk menentukan jumlah cluster optimal.")
        wcss = []
        for i in range(1, 11):
            kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
            kmeans.fit(X)
            wcss.append(kmeans.inertia_)

        fig, ax = plt.subplots()
        ax.plot(range(1, 11), wcss, marker='o', linestyle='--')
        ax.set_title('Elbow Method')
        ax.set_xlabel('Number of clusters')
        ax.set_ylabel('WCSS')
        st.pyplot(fig)

    with tab3:
        st.subheader(f"Visualisasi Cluster (k={k_value})")

        kmeans = KMeans(n_clusters=k_value, init='k-means++', random_state=42)
        y_kmeans = kmeans.fit_predict(X)

        df_result = df.copy()
        df_result['Cluster'] = y_kmeans

        fig_cluster, ax_cluster = plt.subplots(figsize=(10, 6))
        sns.scatterplot(
            x=X.iloc[:, 0], y=X.iloc[:, 1],
            hue=y_kmeans, palette='viridis',
            s=100, ax=ax_cluster, legend='full'
        )

        centers = kmeans.cluster_centers_
        ax_cluster.scatter(centers[:, 0], centers[:, 1], c='red', s=300, alpha=0.5, label='Centroids', marker='X')

        ax_cluster.set_xlabel('Annual Income (k$)')
        ax_cluster.set_ylabel('Spending Score (1-100)')
        ax_cluster.legend()
        st.pyplot(fig_cluster)

        # Download Hasil
        st.subheader("Data Hasil Clustering")
        st.dataframe(df_result, use_container_width=True)

        csv = df_result.to_csv(index=False).encode('utf-8')
        st.download_button(
            label="📥 Download Hasil Clustering (.csv)",
            data=csv,
            file_name='hasil_clustering_mall.csv',
            mime='text/csv',
        )

if __name__ == "__main__":
    main()

Overwriting app_mall.py


In [17]:
from pyngrok import ngrok
ngrok.set_auth_token ("3DHtJAQPSk1YkY163KRRCUHLYo7_3hY3ypFJUgB7vmpuwBAi7")

public_url = ngrok.connect(8501)
print(f"Streamlit app bisa diakses di:\n{public_url}")

!streamlit run app_mall.py &>/dev/null&

Streamlit app bisa diakses di:
NgrokTunnel: "https://freebase-vicinity-dreamily.ngrok-free.dev" -> "http://localhost:8501"
